In [18]:
import serial
from serial.rs485 import RS485
import struct
ser = RS485(port='COM5', baudrate=57600, timeout=1) #'/dev/ttyUSB0'
# Включаем режим RS485, RTS поднимается перед отправкой и опускается после
ser.rs485_mode = serial.rs485.RS485Settings(rts_level_for_tx=True, 
                                            rts_level_for_rx=False)
size = 0
command = bytearray([0x55, 0x55, 0x12, 0x01, 0x84, 0x05, 0x00, 0x08, 0x00, 0x00, 0xFF, 0x00])

def calculate_crc(data):
    
    """Расчет CRC16 для Modbus RTU"""
    crc = 0x0000
    for byte in data:
        crc += byte
    crc -= 0xAA  
    print(crc)
    return crc
    
crc = calculate_crc(command)    
command.append(crc & 0xFF)    

size = command[10]
ser.write(command)                                               #Отправка команды
print(str(command[11])+ str(command[10]))
receive_data = ser.read(size+8) 
data_reg = receive_data[7:7+size]
print(command.hex(' '))
print("Сырые байты:", data_reg.hex(' '))          # Выведет b'\x02\x05\xb6...'
print("В виде чисел:", list(receive_data))   # Выведет [2, 5, 182, ...]
print("Шестнадцатерично:", receive_data.hex(' ')) # Выведет '0205b6...'
# Работаем как обычно
float_value = struct.unpack('<f', receive_data[27:31])
print(float_value)
ser.close()

419
0255
55 55 12 01 84 05 00 08 00 00 ff 00 a3
Сырые байты: 06 00 0f 00 02 00 00 ad 12 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 ed 12 80 41 00 00 00 00 aa 2f 33 40 6d e5 87 40 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 00 80 52 9b 3f e6 01 10 41 18 dc 81 41 aa ac 07 41 8a b7 2c 41 18 dc 81 41 aa ac 07 41 8a b7 2c 41 00 00 00 00 00 00 00 00 80 52 9b 3f e6 01 10 41 50 b0 29 40 50 20 28 40 90 cd 26 40 00 00 04 d4 5d 00 00 d4 5d 00 00 00 00 00 00 01 00 00 00 ff ff 00 00 00 00 00 00 00 00 55 55 55 55 d5 40 00 00 00 00 34 0c 00 00 ff ff ff ff ff ff 10 25 2b 01 1a 01 00 00 e8 51 28 01 23 03 00 00 60 cc 02 00 60 03 00 00 18 05 00 00 17 05 00 00 08 00 00 00 07 06 00 00 00 00 00 00 2f dd e4 3e 00 00 00 00 00 00 00 00 34 03 00 00 2e 02 00 00 1f 02 00 00 1a 02 00 00 15 02 00 00 00 00 00 00 00 00 00 00 10 24 80
В виде чисел: [85, 85, 1, 18, 132, 255, 0, 6, 0, 15, 0, 2, 0, 0, 173, 18, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 237, 18, 128, 65, 0, 0, 0, 0, 170, 47, 51, 64,

In [19]:
float_value = round(struct.unpack('<f', data_reg[27:31])[0],2)
print(data_reg[27:31].hex(' '))#(float_value)
print(float_value)

ed 12 80 41
16.01
